In [ ]:
import os
import pandas as pd

cwd = os.getcwd()

csv_path = os.path.join(cwd, 'review.csv')

df = pd.read_csv(csv_path)
df.columns.to_list()

df = df.drop(columns = ['5',
                        '3',
                        'Unnamed: 3',
                        'museum project, midterm, final.',
                        '2000-01-01 00:00:00'])

df.loc[-1] = df.columns.to_list()
df.index = df.index + 1

df = df.sort_index()
df = df.rename(columns={df.columns[0]: "review", df.columns[1]: "rating"})

Classification
--------------
-> Leverages keyword score + zero-shot classification to classify (on an x-axis):
- about professor
- about course

In [ ]:
#import models
from transformers import pipeline

# X-axis model
focus_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

# Y-axis model
sentiment_classifier = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [67]:
# Classify the focus (prof or course) via zero-shot classification
candidate_labels = [
    "the review evaluates the professor, instructor, teaching style, clarity, personality, helpfulness, or behavior",
    "the review evaluates the course, workload, exams, quizzes, assignments, readings, grading, difficulty, or requirements"
]

def classify_focus(review):
    review = str(review)

    result = focus_classifier(
        review[:1000],
        candidate_labels=candidate_labels,
        hypothesis_template="This student review is mainly about {}."
    )

    scores = dict(zip(result["labels"], result["scores"]))

    professor_prob = scores[candidate_labels[0]]
    course_prob = scores[candidate_labels[1]]

    focus_score = course_prob - professor_prob

    if focus_score < -0.20:
        label = "professor-focused"
    elif focus_score > 0.20:
        label = "course-focused"
    else:
        label = "mixed"

    return pd.Series({
        "professor_prob": professor_prob,
        "course_prob": course_prob,
        "focus_score": focus_score,
        "focus_label": label
    })

In [69]:
df_sample = df.sample(300, random_state=42).copy()

In [71]:
focus_results = df_sample["review"].apply(classify_focus)

df_sample = pd.concat(
    [df_sample.reset_index(drop=True), focus_results.reset_index(drop=True)],
    axis=1
)

In [72]:
#semantic score pipeline
def get_sentiment_score(text):
    result = sentiment_classifier(text[:512])[0]
    label = result["label"].lower()
    score = result["score"]

    if "negative" in label:
        return -score
    elif "positive" in label:
        return score
    else:
        return 0

In [73]:
df_sample["review"] = df_sample["review"].fillna("").astype(str).str.strip()
df_sample["rating"] = pd.to_numeric(df_sample["rating"], errors="coerce")

# drop missing vals
df_sample = df_sample.dropna(subset=["review", "rating"])

# get sentiment score for the extracted sample reviews 
df_sample["semantic_sentiment"] = df_sample["review"].apply(get_sentiment_score)

# converts 1-5 rating score to -1 to + 1
df_sample["rating_score"] = (df_sample["rating"] - 3) / 2

df_sample["y_sentiment_score"] = (
    0.7 * df_sample["semantic_sentiment"] +
    0.3 * df_sample["rating_score"]
)

In [ ]:
# Final plotting columns
df_sample = df_sample.copy()

df_sample["x"] = df_sample["focus_score"]
df_sample["y"] = df_sample["y_sentiment_score"]

In [98]:
# readable hover debug
import textwrap

MAX_CHARS = 350

def shorten_review(text):
    text = str(text).replace("\n", " ").strip()
    
    if len(text) > MAX_CHARS:
        text = text[:MAX_CHARS].rstrip() + "..."
    
    return "<br>".join(textwrap.wrap(text, width=60))

df_sample["review_wrapped"] = df_sample["review"].apply(shorten_review)


In [146]:
# make graph :DDD
import plotly.express as px

fig = px.scatter(
    df_sample,
    x="x",
    y="y",
    color="focus_score",
    color_continuous_scale=[
        [0.0, "#0076B5"],  # professor-focused, Spec blue
        [0.5, "#8C8C8C"],  # mixed
        [1.0, "#D55E00"]   # course-focused, orange
    ],
    range_color=[-1, 1],
    custom_data=["review_wrapped"],
    title=None
)

fig.update_traces(
    hovertemplate=(
        "<b>Review excerpt</b><br><br>"
        "%{customdata[0]}"
        "<extra></extra>"
    ),
    marker=dict(
        size=7,
        opacity=0.72,
        line=dict(width=0.4, color="white")
    )
)

fig.update_layout(
    xaxis_title=None,
    yaxis_title="Review sentiment: negative ← → positive",
    height=560,
    margin=dict(l=80, r=40, t=20, b=95),
    font=dict(
        family="Roboto, Arial, sans-serif",
        size=13,
        color="#253858"
    ),
    paper_bgcolor="white",
    plot_bgcolor="#E8EEF7",
    hoverlabel=dict(
        bgcolor="white",
        font_size=13,
        font_family="Roboto, Arial, sans-serif",
        bordercolor="#253858"
    ),
    coloraxis_colorbar=dict(
        title=dict(text="<b>Review focus</b>", side="top"),
        orientation="h",
        x=0.5,
        xanchor="center",
        y=-0.15,
        yanchor="top",
        len=0.55,
        thickness=14,
        tickmode="array",
        tickvals=[-1, 0, 1],
        ticktext=[
            "Professor",
            "Mixed",
            "Course"
        ]
    )
)

fig.update_xaxes(
    range=[-1, 1],
    zeroline=False,
    showgrid=True,
    gridcolor="white",
    title_standoff=14
)

fig.update_yaxes(
    range=[-1, 1],
    zeroline=False,
    showgrid=True,
    gridcolor="white",
    title_standoff=14
)

fig.add_vline(x=0, line_width=1.2, line_color="#253858")
fig.add_hline(y=0, line_width=1.2, line_color="#253858")

fig.show()

In [147]:
import plotly.io as pio

chart_html = pio.to_html(
    fig,
    full_html=False,
    include_plotlyjs="cdn"
)

In [149]:
html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">

  <title>Are students reviewing the professor or the class?</title>

  <link rel="preconnect" href="https://fonts.googleapis.com">
  <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
  <link href="https://fonts.googleapis.com/css2?family=Roboto:wght@300;400;500;700&display=swap" rel="stylesheet">

  <style>
    body {{
      margin: 0;
      padding: 0;
      font-family: 'Roboto', Arial, sans-serif;
      color: #000000;
      background: #ffffff;
    }}

    .graphic-container {{
      max-width: 1200px;
      margin: 0 auto;
      padding: 24px 20px 32px;
    }}

    h1 {{
      font-family: 'Roboto', Arial, sans-serif;
      font-weight: 700;
      font-size: 28px;
      line-height: 1.15;
      margin: 0 0 8px;
      color: #000000;
    }}

    .dek {{
      font-family: 'Roboto', Arial, sans-serif;
      font-weight: 300;
      font-size: 18px;
      line-height: 1.35;
      margin: 0 0 18px;
      color: #000000;
      max-width: 850px;
    }}

    .chart {{
      width: 100%;
    }}

    .note,
    .source,
    .byline {{
      font-family: 'Roboto', Arial, sans-serif;
      font-weight: 400;
      font-size: 12px;
      line-height: 1.35;
      color: #949494;
      margin-top: 8px;
    }}

    .source a {{
      color: #0076B5;
      text-decoration: none;
    }}

    @media (max-width: 600px) {{
      .graphic-container {{
        padding: 18px 14px 24px;
      }}

      h1 {{
        font-size: 20px;
      }}

     .dek {{
        margin: 0 0 8px;
     }}

      .note,
      .source,
      .byline {{
        font-size: 11px;
      }}
    }}
  </style>
</head>

<body>
  <main class="graphic-container">
    <h1>Are students reviewing the professor or the class?</h1>

    <p class="dek">
      Each dot represents one CULPA review. Reviews farther left focus more on the professor; reviews farther right focus more on the course. Higher dots delineate more positive reviews, while lower dots are more negative.
      In this sample, professor-focused reviews cluster more positively, while course-focused reviews appear more divided.
    </p>

    <section class="chart">
      {chart_html}
    </section>

    <p class="note">
      Note: Analysis uses a random sample of 300 CULPA reviews. Focus and sentiment were estimated using transformer-based text classification.
    </p>

    <p class="source">
      Source: CULPA course reviews
    </p>

    <p class="byline">
      Graphic by Amabelle Alcala
    </p>
  </main>
</body>
</html>
"""

with open("culpa_review_compass_page.html", "w", encoding="utf-8") as f:
    f.write(html)

In [150]:
import os
os.getcwd()

'/Users/mabel/Library/Jupyter/kernels/culpa-story-py311'